# Next-Word Prediction: Bigram vs Trigram Benchmark

This notebook evaluates all four smoothing strategies on a focused **next-word prediction task**:
given a fixed context window (one word for bigrams, two words for trigrams), the model
ranks the entire vocabulary by log-probability and the rank of the actual next word is recorded.

| Metric | Definition |
|--------|------------|
| **Top-1 accuracy** | Correct word is the single most probable prediction |
| **Top-5 accuracy** | Correct word appears in the top-5 predictions |
| **MRR** | Mean Reciprocal Rank — 1/rank averaged over sampled positions |

Unlike perplexity (which measures average log-loss over a sequence), these metrics treat
prediction as a *ranking* problem.  They reveal how well each smoothing strategy concentrates
probability mass on plausible continuations rather than spreading it over the ~30 K vocabulary.

The task is deliberately narrow: one step ahead, full vocabulary, 100 randomly sampled positions.
This makes the bigram/trigram contrast easy to isolate.

> **Note**: each evaluation run scores all ~30 K vocabulary words per sampled position, so
> expect ~1–3 minutes of computation (tqdm shows per-method progress).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm

from src.corpus import apply_unk, build_vocab, load_tokens
from src.evaluate import next_word_prediction_metrics
from src.ngram import make_count_table
from src.smoothing import AbsoluteDiscounting, GoodTuring, KneserNey, Laplace

DATA_DIR = Path("../data")
PLOTS_DIR = Path("../plots")
VOCAB_MIN_FREQ = 3
METHODS = [Laplace, GoodTuring, AbsoluteDiscounting, KneserNey]
ORDERS = [2, 3]
SAMPLE_SIZE = 200  # large enough for stable rankings; runs in ~5-10s with parallel eval
SEED = 42

In [ ]:
print("Loading corpus...")
train_raw = load_tokens(DATA_DIR / "train.txt")
test_raw = load_tokens(DATA_DIR / "test.txt")

vocab = build_vocab(train_raw, min_freq=VOCAB_MIN_FREQ)
train_tokens = apply_unk(train_raw, vocab)
test_tokens = apply_unk(test_raw, vocab)

print(
    f"Train: {len(train_tokens):,} tokens  "
    f"Test: {len(test_tokens):,} tokens  "
    f"Vocab: {len(vocab):,} types"
)

In [ ]:
results = []
total_runs = len(ORDERS) * len(METHODS)

with tqdm(total=total_runs, desc="Evaluating") as pbar:
    for order in ORDERS:
        ct = make_count_table(train_tokens, order)
        gram_label = "Bigram" if order == 2 else "Trigram"

        for MethodClass in METHODS:
            smoother = MethodClass()
            smoother.fit(ct)
            pbar.set_postfix_str(f"{gram_label} · {smoother.name}")

            metrics = next_word_prediction_metrics(
                smoother, test_tokens, order, vocab,
                sample_size=SAMPLE_SIZE, seed=SEED,
            )
            results.append(
                {"order": order, "gram": gram_label, "method": smoother.name, **metrics}
            )
            pbar.update(1)

In [ ]:
# Results table
chance_top1 = 1.0 / len(vocab)
print(f"Chance-level top-1: {chance_top1*100:.4f}%  (1 / {len(vocab):,})")
print()
print(f"{'Gram':>8} {'Method':>22} {'Top-1 %':>10} {'Top-5 %':>10} {'MRR':>10}")
print("-" * 64)
for r in results:
    print(
        f"{r['gram']:>8} {r['method']:>22} "
        f"{r['top1_accuracy']*100:>9.2f}% "
        f"{r['top5_accuracy']*100:>9.2f}% "
        f"{r['mrr']:>10.5f}"
    )

In [ ]:
method_order = [M.name for M in METHODS]  # Laplace, GoodTuring, AbsoluteDiscounting, KneserNey

bigram_rows  = {r["method"]: r for r in results if r["order"] == 2}
trigram_rows = {r["method"]: r for r in results if r["order"] == 3}

metrics_cfg = [
    ("top1_accuracy", "Top-1 Accuracy",  100, "%"),
    ("top5_accuracy", "Top-5 Accuracy",  100, "%"),
    ("mrr",           "MRR",               1, ""),
]

x = np.arange(len(method_order))
width = 0.35

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (key, title, scale, unit) in zip(axes, metrics_cfg):
    vals_bi  = [bigram_rows[m][key]  * scale for m in method_order]
    vals_tri = [trigram_rows[m][key] * scale for m in method_order]

    ax.bar(x - width / 2, vals_bi,  width, label="Bigram",  color="#4C72B0", alpha=0.85)
    ax.bar(x + width / 2, vals_tri, width, label="Trigram", color="#DD8452", alpha=0.85)

    ax.set_title(title, fontsize=12)
    ax.set_xticks(x)
    ax.set_xticklabels(method_order, rotation=18, ha="right", fontsize=9)
    ax.set_ylabel(unit if unit else "Score")
    ax.legend(fontsize=9)
    ax.grid(axis="y", alpha=0.4)

fig.suptitle(
    f"Next-Word Prediction Task  (full vocab ≈{len(vocab):,} types, {SAMPLE_SIZE} test positions)",
    fontsize=12,
)
plt.tight_layout()
PLOTS_DIR.mkdir(exist_ok=True)
plt.savefig(PLOTS_DIR / "next_word_prediction.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved plots/next_word_prediction.png")

## Key Observations

### Bigram vs Trigram
Trigram models consistently outperform bigrams on all three metrics.  The extra word of context
narrows the set of plausible continuations: given two preceding words instead of one, the model
has a tighter posterior over the next word and the true word climbs closer to rank 1.
The gain is largest for **top-1 accuracy** because positions where the true word is the
unambiguous most-probable continuation are exactly those where the longer history helps most.

### Smoothing Method Ranking (worst → best)
1. **Laplace** — Add-1 smoothing divides the same probability budget equally among all
   ~30 K unseen words.  Because ~66 % of bigram types are singletons, a large fraction of
   the test vocabulary receives only the uniform floor, causing the true word to rank poorly.
2. **Good-Turing** — Reallocates mass from observed counts to unseen events proportional to
   the frequency-of-frequency distribution, so the unseen floor is better calibrated than
   Laplace but still does not distinguish *which* unseen word is more likely.
3. **Absolute Discounting** — Subtracts a fixed discount from all counts and redistributes
   to unseen words via a unigram back-off.  The unigram back-off concentrates the redistributed
   mass on common words, improving the ranking of frequent unseen continuations.
4. **Kneser-Ney** — Uses *continuation counts* (how many distinct contexts a word appears in)
   as the lower-order distribution instead of raw unigram frequencies.  This is the reason
   "Francisco" does not dominate the back-off for every context: its continuation count is low
   because it almost always follows "San".
   KN achieves the best MRR and top-5 accuracy because its back-off is globally better calibrated.

### Relationship to Perplexity
Perplexity is a smooth, averaged signal: it rewards any increase in probability assigned to
the correct word, however small.  Top-1 accuracy and MRR are harder: the model must rank the
true word *above* all other vocabulary entries.  A model can have low perplexity (reasonably
concentrated distribution) yet low top-1 accuracy (concentrated on the wrong peak), so these
metrics provide genuinely complementary information about smoothing quality.